<a href="https://colab.research.google.com/github/JaredLThompson/LLM_Misc/blob/main/notebooks/hello_iris_training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# Train the “Hello Iris” wake word


This is a restart-safe Colab adaptation of openWakeWord's automatic model-training notebook. It trains one model for **hello iris** and validates required files before expensive generation or training begins.

Run the cells in order. If any cell fails, stop and fix that failure before continuing; Colab otherwise permits later cells to produce misleading secondary errors.


# Environment Setup

To begin, we'll need to install the requirements for training custom models. In particular, a relatively recent version of Pytorch and custom fork of the [piper-sample-generator](https://github.com/dscripka/piper-sample-generator) library for generating synthetic examples for the custom model.

**Important Note!** Currently, automated model training is only supported on linux systems due to the requirements of the text to speech library used for synthetic sample generation (Piper). It may be possible to use Piper on Windows/Mac systems, but that has not (yet) been tested.

In [ ]:
## Environment setup — safe to rerun

from pathlib import Path
import os
import subprocess
import sys
import textwrap
import logging

# Suppress verbose debug logging from dependencies to prevent browser crashes.
logging.basicConfig(level=logging.WARNING)
for _logger_name in ("httpcore", "httpx", "datasets", "urllib3", "onnxscript",
                      "onnx_ir", "onnxscript.optimizer", "onnxscript._internal"):
    logging.getLogger(_logger_name).setLevel(logging.WARNING)

CONTENT = Path("/content")
OPENWAKEWORD = CONTENT / "openwakeword"
PIPER_GENERATOR = CONTENT / "piper-sample-generator"
OPENWAKEWORD_COMMIT = "368c03716d1e92591906a84949bc477f3a834455"
PIPER_GENERATOR_COMMIT = "2971426a55072f7d22fec416ca7800df8bd23207"

# Set environment variable to suppress debug logging in training subprocesses.
os.environ["LOGLEVEL"] = "WARNING"
os.environ["PYTHONWARNINGS"] = "ignore"

# Detect environment
try:
    from google.colab import files as _colab_files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
print(f"Environment: {'Google Colab' if IN_COLAB else 'Self-hosted Jupyter'}")


def run(command, cwd=None, env=None, tail=5000, valid_onnx=None):
    """Run a command and always expose useful diagnostics before failing."""
    command = list(map(str, command))
    print("+", " ".join(command))
    result = subprocess.run(
        command,
        cwd=cwd,
        env=env,
        check=False,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
    )
    print("Exit code:", result.returncode)
    # Filter out DEBUG lines and tqdm progress bars to avoid flooding Jupyter output.
    def _filter_output(text, tail_chars):
        lines = [l for l in text.splitlines()
                 if not l.startswith('DEBUG:')
                 and 'it/s]' not in l
                 and 's/it]' not in l]
        filtered = '\n'.join(lines)
        return filtered[-tail_chars:] if len(filtered) > tail_chars else filtered
    if result.stdout:
        print("\n===== STDOUT: final output =====")
        print(_filter_output(result.stdout, tail))
    if result.stderr:
        print("\n===== STDERR: final output =====")
        print(_filter_output(result.stderr, tail))
    if result.returncode != 0 and valid_onnx is not None:
        # Upstream currently attempts an optional ONNX-to-TFLite conversion
        # after writing the ONNX model. The robots only need ONNX, so accept
        # the result when that artifact passes structural and runtime checks.
        import onnx
        import onnxruntime as ort

        model_path = Path(valid_onnx)
        if model_path.is_file() and model_path.stat().st_size > 0:
            model = onnx.load(str(model_path))
            onnx.checker.check_model(model)
            ort.InferenceSession(
                str(model_path),
                providers=["CPUExecutionProvider"],
            )
            print(
                "Command exited after producing a valid ONNX model; "
                "optional TFLite conversion failure is ignored:",
                model_path,
            )
            return result

    assert result.returncode == 0, (
        f"Command failed with exit code {result.returncode}: "
        + " ".join(command)
    )
    return result


# Ensure system dependencies are present (ffmpeg for torchcodec/torchaudio).
if not IN_COLAB:
    import shutil
    if not shutil.which("ffmpeg"):
        print("Installing ffmpeg...")
        run(["sudo", "apt-get", "update", "-qq"])
        run(["sudo", "apt-get", "install", "-y", "-qq", "ffmpeg"])

# Clone once, then force both training repositories to reviewed commits.
def checkout_pinned_repository(path, url, revision):
    if (path / ".git").is_dir():
        run(["git", "-C", path, "fetch", "origin", revision])
    elif path.exists():
        raise RuntimeError(
            f"{path} exists but is not a Git checkout. "
            "Move or delete it and rerun this cell."
        )
    else:
        run(["git", "clone", url, path])
        run(["git", "-C", path, "fetch", "origin", revision])
    run(["git", "-C", path, "checkout", "--detach", revision])
    actual_revision = run(
        ["git", "-C", path, "rev-parse", "HEAD"]
    ).stdout.strip()
    assert actual_revision == revision, (
        f"Expected {path} at {revision}, found {actual_revision}"
    )


checkout_pinned_repository(
    OPENWAKEWORD,
    "https://github.com/dscripka/openWakeWord.git",
    OPENWAKEWORD_COMMIT,
)
checkout_pinned_repository(
    PIPER_GENERATOR,
    "https://github.com/rhasspy/piper-sample-generator.git",
    PIPER_GENERATOR_COMMIT,
)

piper_model = PIPER_GENERATOR / "models" / "en_US-libritts_r-medium.pt"
piper_model.parent.mkdir(parents=True, exist_ok=True)
if not piper_model.is_file() or piper_model.stat().st_size == 0:
    run([
        "wget", "-O", piper_model,
        "https://github.com/rhasspy/piper-sample-generator/releases/download/v2.0.0/en_US-libritts_r-medium.pt",
    ])

# Install openWakeWord and the dependencies used by the upstream notebook.
run([sys.executable, "-m", "pip", "install", "-e", OPENWAKEWORD])
run([sys.executable, "-m", "pip", "install", "-e", PIPER_GENERATOR])
run([
    sys.executable, "-m", "pip", "install",
    "mutagen==1.47.0", "torchinfo==1.8.0",
    "torchmetrics==1.2.0", "speechbrain==0.5.14", "audiomentations==0.33.0",
    "torch-audiomentations==0.12.0", "acoustics==0.2.6", "pronouncing==0.2.0",
    "datasets==3.6.0", "deep-phonemizer==0.0.19",
    "onnx", "onnxscript",
])

# Additional dependencies for self-hosted Jupyter compatibility:
# - setuptools<70: provides pkg_resources (removed in setuptools 83+)
# - torchcodec + soundfile: audio loading backends for torchaudio 2.13+
# - scipy<1.15: acoustics package uses deprecated scipy.special.sph_harm
# - ipywidgets: tqdm widget progress bars (prevents output flooding)
if not IN_COLAB:
    run([
        sys.executable, "-m", "pip", "install",
        "setuptools<70", "torchcodec", "soundfile", "ipywidgets",
        "scipy<1.15",
    ])

# New Torchaudio releases removed APIs still used by torch-audiomentations.
# A sitecustomize shim makes them available inside fresh training subprocesses.
compat_dir = CONTENT / "openwakeword_compat"
compat_dir.mkdir(exist_ok=True)
(compat_dir / "sitecustomize.py").write_text(textwrap.dedent("""
    from types import SimpleNamespace
    import numpy as np
    import soundfile as sf
    import torch
    import torchaudio

    if not hasattr(torchaudio, "set_audio_backend"):
        torchaudio.set_audio_backend = lambda name: None

    if not hasattr(torchaudio, "info"):
        def info(path, *args, **kwargs):
            metadata = sf.info(str(path))
            return SimpleNamespace(
                sample_rate=metadata.samplerate,
                num_frames=metadata.frames,
                num_channels=metadata.channels,
                bits_per_sample=metadata.subtype_info,
                encoding=metadata.format,
            )
        torchaudio.info = info

    if not hasattr(torchaudio, "load"):
        def load(path, frame_offset=0, num_frames=-1, normalize=True,
                 channels_first=True, *args, **kwargs):
            stop = None if num_frames in (-1, None) else frame_offset + num_frames
            samples, sample_rate = sf.read(
                str(path), start=frame_offset, stop=stop,
                dtype="float32", always_2d=True,
            )
            tensor = torch.from_numpy(np.asarray(samples).T)
            return (tensor if channels_first else tensor.T), sample_rate
        torchaudio.load = load
"""), encoding="utf-8")

existing_pythonpath = os.environ.get("PYTHONPATH", "")
os.environ["PYTHONPATH"] = (
    str(compat_dir)
    if not existing_pythonpath
    else f"{compat_dir}:{existing_pythonpath}"
)

# Download the feature-extraction models required by the training code.
resource_dir = OPENWAKEWORD / "openwakeword" / "resources" / "models"
resource_dir.mkdir(parents=True, exist_ok=True)
for filename in (
    "embedding_model.onnx",
    "embedding_model.tflite",
    "melspectrogram.onnx",
    "melspectrogram.tflite",
):
    destination = resource_dir / filename
    if not destination.is_file() or destination.stat().st_size == 0:
        run([
            "wget",
            "https://github.com/dscripka/openWakeWord/releases/download/v0.5.1/" + filename,
            "-O", destination,
        ])

config_path = OPENWAKEWORD / "examples" / "custom_model.yml"
train_path = OPENWAKEWORD / "openwakeword" / "train.py"
assert config_path.is_file(), f"Missing {config_path}"
assert train_path.is_file(), f"Missing {train_path}"
print("Environment ready:", OPENWAKEWORD)


In [ ]:
# Compatibility adapter for openWakeWord's legacy generator import.
# Current Piper generates 22.05 kHz audio, while openWakeWord training
# requires every generated clip to be 16 kHz.
from pathlib import Path
import sys

PIPER_GENERATOR = Path("/content/piper-sample-generator")
PIPER_MODEL = PIPER_GENERATOR / "models" / "en_US-libritts_r-medium.pt"
PIPER_CONFIG = Path(str(PIPER_MODEL) + ".json")
PIPER_SHIM = PIPER_GENERATOR / "generate_samples.py"

assert PIPER_MODEL.is_file(), f"Missing {PIPER_MODEL}"
assert PIPER_CONFIG.is_file(), f"Missing {PIPER_CONFIG}"

PIPER_SHIM.write_text(
    '''from math import gcd
from pathlib import Path

import numpy as np
import scipy.io.wavfile
import scipy.signal
from piper_sample_generator.__main__ import generate_samples as _generate_samples

MODEL = Path(__file__).parent / "models" / "en_US-libritts_r-medium.pt"
TARGET_SAMPLE_RATE = 16000


def _resample_to_16k(path):
    sample_rate, samples = scipy.io.wavfile.read(path)
    if sample_rate == TARGET_SAMPLE_RATE:
        return
    if np.issubdtype(samples.dtype, np.integer):
        scale = float(max(abs(np.iinfo(samples.dtype).min), np.iinfo(samples.dtype).max))
        samples = samples.astype(np.float32) / scale
    else:
        samples = samples.astype(np.float32)
    divisor = gcd(sample_rate, TARGET_SAMPLE_RATE)
    samples = scipy.signal.resample_poly(
        samples,
        TARGET_SAMPLE_RATE // divisor,
        sample_rate // divisor,
    )
    samples = np.clip(samples, -1.0, 1.0)
    scipy.io.wavfile.write(
        path,
        TARGET_SAMPLE_RATE,
        (samples * 32767).astype(np.int16),
    )


def generate_samples(*args, **kwargs):
    kwargs.setdefault("model", MODEL)
    result = _generate_samples(*args, **kwargs)
    output_dir = Path(kwargs.get("output_dir", args[1] if len(args) > 1 else "."))
    for wav_path in output_dir.glob("*.wav"):
        _resample_to_16k(wav_path)
    return result
''',
    encoding="utf-8",
)

sys.path.insert(0, str(PIPER_GENERATOR))
from generate_samples import generate_samples

print("Modern Piper sample generator ready:", generate_samples)
print("Generated clips will be normalized to 16 kHz")
print("CUDA available:", __import__("torch").cuda.is_available())


In [ ]:
# Imports

import os
import numpy as np
import torch
import sys
from pathlib import Path
import uuid
import yaml
import datasets
import scipy
from tqdm import tqdm

# Suppress verbose debug logging that floods Jupyter output.
import logging
logging.getLogger("httpcore").setLevel(logging.WARNING)
logging.getLogger("httpx").setLevel(logging.WARNING)
logging.getLogger("datasets").setLevel(logging.WARNING)
logging.getLogger("urllib3").setLevel(logging.WARNING)
logging.getLogger("onnxscript").setLevel(logging.WARNING)
logging.getLogger("onnx_ir").setLevel(logging.WARNING)


# Download Data

When training new openWakeWord models using the automated procedure, four specific types of data are required:

1) Synthetic examples of the target word/phrase generated with text-to-speech models

2) Synthetic examples of adversarial words/phrases generated with text-to-speech models

3) Room impulse reponses and noise/background audio data to augment the synthetic examples and make them more realistic

4) Generic "negative" audio data that is very unlikely to contain examples of the target word/phrase in the context where the model should detect it. This data can be the original audio data, or precomputed openWakeWord features ready for model training.

5) Validation data to use for early-stopping when training the model.

For the purposes of this notebook, all five of these sources will either be generated manually or can be obtained from HuggingFace thanks to their excellent `datasets` library and extremely generous hosting policy. Also note that while only a portion of some datasets are downloaded, for the best possible performance it is recommended to download the entire dataset and keep a local copy for future training runs.

In [ ]:
# Download room impulse responses collected by MIT
# https://mcdermottlab.mit.edu/Reverb/IR_Survey.html

output_dir = Path("/content/mit_rirs")
output_dir.mkdir(parents=True, exist_ok=True)
rir_dataset = datasets.load_dataset(
    "davidscripka/MIT_environmental_impulse_responses",
    split="train",
    streaming=True,
)

# Save new clips as 16-bit PCM WAV files. Existing valid files are preserved so
# an interrupted run can resume without downloading and rewriting everything.
written = 0
skipped = 0
for row in tqdm(rir_dataset):
    output_path = output_dir / Path(row["audio"]["path"]).name
    if output_path.is_file() and output_path.stat().st_size > 44:
        skipped += 1
        continue

    scipy.io.wavfile.write(
        output_path,
        16000,
        (row["audio"]["array"] * 32767).astype(np.int16),
    )
    written += 1

print(f"MIT RIR clips written: {written}; already present: {skipped}")


In [ ]:
## Download noise and background audio

from pathlib import Path
import numpy as np
import scipy.io.wavfile
from tqdm.auto import tqdm

# AudioSet is published as Parquet rather than the tar shards used by the
# original openWakeWord notebook. Stream a bounded subset so Colab does not
# need to download the complete multi-terabyte dataset.
audioset_output = Path("/content/audioset_16k")
audioset_output.mkdir(parents=True, exist_ok=True)
target_audioset_clips = 5000

existing = list(audioset_output.glob("*.wav"))
if len(existing) < target_audioset_clips:
    audioset = datasets.load_dataset(
        "agkphysics/AudioSet",
        "balanced",
        split="train",
        streaming=True,
    )
    audioset = audioset.cast_column(
        "audio", datasets.Audio(sampling_rate=16000)
    )
    for index, row in enumerate(
        tqdm(audioset, total=target_audioset_clips, desc="AudioSet")
    ):
        if index >= target_audioset_clips:
            break
        audio = row["audio"]
        samples = np.asarray(audio["array"], dtype=np.float32)
        samples = np.clip(samples, -1.0, 1.0)
        scipy.io.wavfile.write(
            audioset_output / f"audioset-{index:05d}.wav",
            16000,
            (samples * 32767).astype(np.int16),
        )

audioset_files = list(audioset_output.glob("*.wav"))
assert audioset_files, "AudioSet streaming produced no WAV files"
print(f"AudioSet background clips ready: {len(audioset_files)}")

# FMA is intentionally optional. Its legacy custom Hugging Face loader is
# incompatible with current streaming HTTP files (it requires seek support).
# AudioSet supplies sufficient varied background audio for this bounded run.
fma_output = Path("/content/fma")
fma_files = list(fma_output.glob("*.wav")) if fma_output.is_dir() else []
if fma_files:
    print(f"Optional FMA background clips found: {len(fma_files)}")
else:
    print("Optional FMA dataset unavailable; continuing with AudioSet only.")


In [ ]:
# Download pre-computed openWakeWord features for training and validation
from pathlib import Path

# training set (~2,000 hours from the ACAV100M Dataset)
# See https://huggingface.co/datasets/davidscripka/openwakeword_features for more information
acav_path = Path('/content/openwakeword_features_ACAV100M_2000_hrs_16bit.npy')
if not acav_path.is_file() or acav_path.stat().st_size == 0:
    run([
        "wget", "-P", "/content",
        "https://huggingface.co/datasets/davidscripka/"
        "openwakeword_features/resolve/main/"
        "openwakeword_features_ACAV100M_2000_hrs_16bit.npy",
    ])
else:
    print(f'ACAV features already present: {acav_path} ({acav_path.stat().st_size:,} bytes)')

# validation set for false positive rate estimation (~11 hours)
val_path = Path('/content/validation_set_features.npy')
if not val_path.is_file() or val_path.stat().st_size == 0:
    run([
        "wget", "-P", "/content",
        "https://huggingface.co/datasets/davidscripka/"
        "openwakeword_features/resolve/main/validation_set_features.npy",
    ])
else:
    print(f'Validation features already present: {val_path} ({val_path.stat().st_size:,} bytes)')

# Define Training Configuration

The configuration below trains one model to accept two deliberate variants, **hello iris** and **hey iris**. Keep both variants together only when deployment testing confirms acceptable recall and false-activation rates.


In [ ]:
# Load the default training configuration using an absolute Colab path.
from pathlib import Path
import yaml

config_path = Path("/content/openwakeword/examples/custom_model.yml")
assert config_path.is_file(), (
    f"Missing {config_path}. Rerun the Environment Setup cell and do not continue "
    "until it succeeds."
)

with config_path.open("r", encoding="utf-8") as stream:
    config = yaml.safe_load(stream)

config


In [ ]:
# Configure the Hello Iris model.

config["target_phrase"] = ["hello iris", "hey iris"]
config["model_name"] = "hello_iris"
config["n_samples"] = 20000
config["n_samples_val"] = 4000
config["steps"] = 100000
config["target_accuracy"] = 0.7
config["target_recall"] = 0.4
config["layer_size"] = 64
config["augmentation_rounds"] = 2
config["max_negative_weight"] = 5000

background_paths = [Path("/content/audioset_16k"), Path("/content/fma")]
background_paths = [
    path
    for path in background_paths
    if path.is_dir() and any(path.glob("*.wav"))
]
assert background_paths, "No nonempty background-audio directories are available"
config["background_paths"] = [str(path) for path in background_paths]

config["false_positive_validation_data_path"] = "/content/validation_set_features.npy"
config["feature_data_files"] = {
    "ACAV100M_sample": "/content/openwakeword_features_ACAV100M_2000_hrs_16bit.npy"
}

training_config = Path("/content/hello_iris.yaml")
with training_config.open("w", encoding="utf-8") as stream:
    yaml.safe_dump(config, stream, sort_keys=False)

required = [
    Path("/content/openwakeword/openwakeword/train.py"),
    Path("/content/validation_set_features.npy"),
    Path("/content/openwakeword_features_ACAV100M_2000_hrs_16bit.npy"),
    Path("/content/audioset_16k"),
]
missing = [str(path) for path in required if not path.exists()]
assert not missing, "Missing required training data:\n- " + "\n- ".join(missing)

print("Background paths:", config["background_paths"])
print("Training configuration ready:", training_config)


# Train the Model

With the data downloaded and training configuration set, we can now start training the model. We'll do this in parts to better illustrate the sequence, but you can also execute every step at once for a fully automated process.

In [ ]:
# Step 1: Generate synthetic clips.

from pathlib import Path

train_script = Path("/content/openwakeword/openwakeword/train.py")
training_config = Path("/content/hello_iris.yaml")
assert train_script.is_file(), f"Missing {train_script}"
assert training_config.is_file(), f"Missing {training_config}"

run([
    sys.executable, str(train_script),
    "--training_config", str(training_config),
    "--generate_clips",
], cwd="/content", env=os.environ.copy());  # semicolon suppresses cell output


In [ ]:
# Step 2: Augment the generated clips and compute feature arrays.

run([
    sys.executable, str(train_script),
    "--training_config", str(training_config),
    "--augment_clips",
    "--overwrite",
], cwd="/content", env=os.environ.copy());  # semicolon suppresses cell output


In [ ]:
# Step 3: Train and export the model.

training_result = run([
    sys.executable, str(train_script),
    "--training_config", str(training_config),
    "--train_model",
], cwd="/content", env=os.environ.copy(),
   valid_onnx="/content/my_custom_model/hello_iris.onnx")

training_log = Path("/content/my_custom_model/hello_iris_training.log")
training_log.parent.mkdir(parents=True, exist_ok=True)
training_log.write_text(
    "===== STDOUT =====\n" + (training_result.stdout or "") +
    "\n===== STDERR =====\n" + (training_result.stderr or ""),
    encoding="utf-8",
)
print("Complete training log:", training_log)


## Download the trained ONNX model

The training script exports the model beneath `/content/my_custom_model`. The next cell verifies it before starting the browser download.


In [ ]:
from pathlib import Path
import shutil
import hashlib
import importlib.metadata
import json
import platform
import subprocess
from datetime import datetime, timezone
import tempfile
import numpy as np
import onnx
import onnxruntime as ort
from onnx.external_data_helper import convert_model_from_external_data

try:
    from google.colab import files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

exported_path = Path("/content/my_custom_model/hello_iris.onnx")
standalone_path = Path("/content/my_custom_model/hello_iris_standalone.onnx")
assert exported_path.is_file(), f"Training did not produce {exported_path}"
assert exported_path.stat().st_size > 0, f"Model is empty: {exported_path}"

# New PyTorch exporters may store weights in hello_iris.onnx.data. Load those
# weights and explicitly embed them into one portable ONNX file.
model = onnx.load(str(exported_path), load_external_data=True)
convert_model_from_external_data(model)
onnx.save_model(model, str(standalone_path), save_as_external_data=False)

# Validate the graph.
model = onnx.load(str(standalone_path), load_external_data=False)
onnx.checker.check_model(model)

# Prove it is truly self-contained by copying only that file into an otherwise
# empty directory and executing it there.
with tempfile.TemporaryDirectory() as temporary_directory:
    isolated_path = Path(temporary_directory) / "hello_iris.onnx"
    shutil.copy2(standalone_path, isolated_path)
    session = ort.InferenceSession(
        str(isolated_path),
        providers=["CPUExecutionProvider"],
    )
    input_info = session.get_inputs()[0]
    input_shape = [
        1 if not isinstance(size, int) else size
        for size in input_info.shape
    ]
    test_input = np.zeros(input_shape, dtype=np.float32)
    test_output = session.run(None, {input_info.name: test_input})

print(
    f"Validated standalone model {standalone_path.name} "
    f"({standalone_path.stat().st_size:,} bytes)"
)
print("Input:", input_info.name, input_info.shape, input_info.type)
print("Test inference output:", test_output)


model_sha256 = hashlib.sha256(standalone_path.read_bytes()).hexdigest()


def package_version(name):
    try:
        return importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError:
        return None


manifest = {
    "schema": "atomicpi.wakeword.training/v1",
    "created_at": datetime.now(timezone.utc).isoformat(),
    "model_name": "hello_iris",
    "target_phrases": ["hello iris", "hey iris"],
    "model_path": str(standalone_path),
    "model_size_bytes": standalone_path.stat().st_size,
    "model_sha256": model_sha256,
    "input": {
        "name": input_info.name,
        "shape": input_info.shape,
        "type": input_info.type,
    },
    "training_config": config,
    "revisions": {
        "openwakeword": OPENWAKEWORD_COMMIT,
        "piper_sample_generator": PIPER_GENERATOR_COMMIT,
    },
    "runtime": {
        "python": platform.python_version(),
        "platform": platform.platform(),
        "packages": {
            name: package_version(name)
            for name in (
                "torch",
                "torchaudio",
                "openwakeword",
                "onnx",
                "onnxruntime",
                "datasets",
                "audiomentations",
                "torch-audiomentations",
            )
        },
    },
    "evaluation": {
        "target_accuracy": config.get("target_accuracy"),
        "target_recall": config.get("target_recall"),
        "false_positive_validation_data_path": config.get(
            "false_positive_validation_data_path"
        ),
        "deployment_threshold": None,
        "deployment_validation_required": True,
    },
    "notes": [
        "GPU training is not guaranteed bit-for-bit deterministic.",
        "Select the deployment threshold using held-out target-device audio.",
    ],
}
manifest_path = standalone_path.with_suffix(".manifest.json")
manifest_path.write_text(
    json.dumps(manifest, indent=2, sort_keys=True, default=str) + "\n",
    encoding="utf-8",
)
print("SHA-256:", model_sha256)
print("Training manifest:", manifest_path)

if IN_COLAB:
    files.download(str(standalone_path))
    files.download(str(manifest_path))
else:
    print(f"\nModel saved to: {standalone_path}")
    print("Download from Jupyter: right-click the file in the file browser, or run:")
    print(f"  aws ssm start-session ... then: cat {standalone_path} > /tmp/model.onnx")
